# CS F425: Deep Learning Course Project (2026–27)
## Task-II: Additional Task 3 — Absolute Mask Difference (|VH - VV|) in dB Scale

**Team 26**  
- **Aarnav Piparsania** (2024ADPS0729P)  
- **Samarth Soni** (2024A2PS1243P)  
- **Bhavya Gupta** (2023A7PS0494P)  

**District:** Chhatarpur, Madhya Pradesh  
**Flood Event:** 29 August – 3 September 2026  
**Deliverable Output:** `Mask_diff_dB.tif`

---
### 1. Theoretical Background & Objectives
The absolute difference between cross-polarization (VH) and co-polarization (VV) flood masks:

$$\text{Mask}_{\text{diff}} = \left| \text{Mask}_{\text{VH}} - \text{Mask}_{\text{VV}} \right|$$

isolates the exact geographic zones of **polarimetric disagreement**:
1. **$\text{Mask}_{\text{diff}} = 0$ (Agreement):** Both polarizations agree either that the pixel is unflooded land, or that it is fully submerged open water.
2. **$\text{Mask}_{\text{diff}} = 1$ (Disagreement):** Points to complex scattering regimes such as partially submerged vegetation, wet soil transition zones, or urban/settlement boundaries where double-bounce and volume scattering behave differently between horizontal and vertical polarizations.

In [ ]:
# Step 1: Import Dependencies
import os
import numpy as np
import tifffile
import matplotlib.pyplot as plt

print("Dependencies loaded.")

In [ ]:
# Step 2: Define Paths & Load Masks
base_dir = os.path.abspath("..")
proc_dir = os.path.join(base_dir, "Processed data", "Chhatarpur")
mask_dir = os.path.join(proc_dir, "Mask", "Task images")

mask_vh_path = os.path.join(proc_dir, "Mask", "Cleaned_flood_mask.tif")
mask_or_path = os.path.join(mask_dir, "Mask_OR_dB.tif")
output_diff_mask = os.path.join(mask_dir, "Mask_diff_dB.tif")

mask_vh = (tifffile.imread(mask_vh_path) > 0).astype(np.uint8)
mask_or = (tifffile.imread(mask_or_path) > 0).astype(np.uint8)

# Derive VV mask from OR relationship
mask_vv = (mask_or ^ mask_vh) | (mask_vh & mask_or)

In [ ]:
# Step 3: Compute Absolute Mask Difference |VH - VV|
mask_diff = np.abs(mask_vh.astype(np.int8) - mask_vv.astype(np.int8)).astype(np.uint8)

mask_diff_export = mask_diff * 255
tifffile.imwrite(output_diff_mask, mask_diff_export, compression='zlib')
print(f"Successfully exported {output_diff_mask}!")

diff_pixels = np.count_nonzero(mask_diff == 1)
total_flood_pixels = np.count_nonzero(mask_or == 1)
disagreement_rate = (diff_pixels / total_flood_pixels * 100.0) if total_flood_pixels > 0 else 0

print(f"Disagreement Pixels: {diff_pixels:,}")
print(f"Polarization Disagreement Rate across Inundated Area: {disagreement_rate:.2f}%")

In [ ]:
# Step 4: Visualize Polarimetric Disagreement Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
stride = 4

axes[0].imshow(mask_vh[::stride, ::stride], cmap='Blues')
axes[0].set_title("VH Inundation Mask")
axes[0].axis("off")

axes[1].imshow(mask_vv[::stride, ::stride], cmap='Greens')
axes[1].set_title("VV Inundation Mask")
axes[1].axis("off")

axes[2].imshow(mask_diff_export[::stride, ::stride], cmap='hot')
axes[2].set_title("Mask Difference |VH - VV| (Mask_diff_dB.tif)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Discussion & Findings for Task 3
- **Boundary & Transition Sensitivity:** Pixels where $|\text{Mask}_{\text{VH}} - \text{Mask}_{\text{VV}}| = 1$ predominantly concentrate along floodwater boundaries, shallow seasonal drainage swales, and vegetated river embankments.
- **Vegetation Canopy Scattering:** VH detects flood beneath low vegetation canopy where depolarization drops, whereas VV may detect surface water in open soil patches with minimal vegetation. This difference maps boundary uncertainty and vegetation-water interactions along the Dhasan and Ken river basins.